# Notebook 04 — Malaysian Data Pipeline

This notebook produces two output datasets:

- **Pipeline A** — `data/malaysian_gold_standard_unverified.csv` — 150 stratified rows with candidate pipeline labels. You manually verify these to produce `malaysian_gold_standard.csv`.
- **Pipeline B** — `data/dashboard_data.csv` — 300 rows per restaurant (random sample), run through the pipeline in auto mode, exploded to one row per aspect-sentiment pair.

Run cells in order. Do not skip the path setup cell.

## 0. Environment Setup

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))  # MUST be before any src imports

import pandas as pd
import numpy as np
from pipeline import run_pipeline
from categoriser import AspectCategoriser

import spacy
nlp = spacy.load('en_core_web_md')

print('Imports OK')

Imports OK


## 1. Load Raw Data

In [3]:
df_raw = pd.read_csv('../data/All_Restaurants.csv')
print('Raw shape:', df_raw.shape)
df_raw.head(3)

Raw shape: (1362, 8)


,place_name,review_id,rating,review_text,original_language,published_at,review_translated_text,translated_language
0,Raihana One Bistro,Ci9DQUlRQUNvZENodHljRjlvT2xOMWRXZExRazFDZVMxU0...,2,"banyak kali , kalau order take away jeeee, cak...",ms,4 bulan yang lalu,NaN,NaN
1,Raihana One Bistro,Ci9DQUlRQUNvZENodHljRjlvT2xJM1l6ZDRaSE5PUVdkbV...,5,"Im regular here\r\nMy fav is tosak, telur sepa...",ms,6 bulan yang lalu,NaN,NaN
2,Raihana One Bistro,ChZDSUhNMG9nS0VJQ0FnSUNib1lMWEVREAE,3,"27/7/24, Saturday, 10.20am:\r\n\r\nBrunch sini...",ms,setahun yang lalu,NaN,NaN


## 2. Clean — Language Filter + Drop Short Reviews

In [4]:
# Keep Malay, English, Indonesian (Indonesian detected as BM by Google — known issue)
KEEP_LANGS = ['ms', 'en', 'id']
df = df_raw[df_raw['original_language'].isin(KEEP_LANGS)].copy()

# Drop reviews under 10 characters
df = df[df['review_text'].str.len() >= 10].copy()

# Reset index
df = df.reset_index(drop=True)

print('After cleaning:', df.shape)
print()
print('Per restaurant:')
print(df['place_name'].value_counts())
print()
print('Language breakdown per restaurant:')
print(df.groupby(['place_name', 'original_language']).size().unstack(fill_value=0))

After cleaning: (1310, 8)

Per restaurant:
place_name
SABA Restaurant Cyberjaya    493
Jom Corner (J Corner)        486
Raihana One Bistro           331
Name: count, dtype: int64

Language breakdown per restaurant:
original_language           en  id   ms
place_name                             
Jom Corner (J Corner)      242  35  209
Raihana One Bistro         205  13  113
SABA Restaurant Cyberjaya    0   0  493


## 3. Pipeline A — Stratified Sampling (50 rows per restaurant)

Stratification logic per restaurant:
- **Group A** — English or mixed (`original_language == 'en'`): 20 rows
- **Group B** — Primarily Malay (`original_language == 'ms'` or `'id'`): 20 rows  
- **Group C** — Short / ambiguous (10–30 chars, any language): 10 rows

Within each group, rows are sampled with a fixed random seed for reproducibility.

In [5]:
SEED = 42

def stratified_sample(restaurant_df, n_english=20, n_malay=20, n_short=10):
    """
    Returns 50 stratified rows for gold standard annotation.
    Group C (short) is sampled first to avoid overlap with A and B.
    If a restaurant has no English rows, the English quota is redistributed to Malay.
    """
    df = restaurant_df.copy()
    
    # Group C — short reviews (10–30 chars), any language
    short_mask = df['review_text'].str.len().between(10, 30)
    group_c = df[short_mask].sample(min(n_short, short_mask.sum()), random_state=SEED)
    remaining = df[~df.index.isin(group_c.index)]
    
    # Group A — English
    eng_mask = remaining['original_language'] == 'en'
    available_english = eng_mask.sum()
    group_a = remaining[eng_mask].sample(min(n_english, available_english), random_state=SEED)
    remaining = remaining[~remaining.index.isin(group_a.index)]
    
    # Redistribute unfilled English slots to Malay
    english_shortfall = n_english - len(group_a)
    n_malay_adjusted = n_malay + english_shortfall
    
    # Group B — Malay / Indonesian (absorbs English shortfall)
    malay_mask = remaining['original_language'].isin(['ms', 'id'])
    group_b = remaining[malay_mask].sample(min(n_malay_adjusted, malay_mask.sum()), random_state=SEED)
    
    sample = pd.concat([group_a, group_b, group_c]).drop_duplicates()
    sample['strata'] = (
        ['english'] * len(group_a) +
        ['malay'] * len(group_b) +
        ['short'] * len(group_c)
    )
    return sample


restaurants = df['place_name'].unique()
gold_frames = []

for restaurant in restaurants:
    r_df = df[df['place_name'] == restaurant]
    sample = stratified_sample(r_df)
    gold_frames.append(sample)
    print(f'{restaurant}: {len(sample)} rows sampled')
    print(sample['strata'].value_counts())
    print()

gold_df = pd.concat(gold_frames).reset_index(drop=True)
print('Total gold standard rows:', len(gold_df))

Raihana One Bistro: 50 rows sampled
strata
english    20
malay      20
short      10
Name: count, dtype: int64

SABA Restaurant Cyberjaya: 50 rows sampled
strata
malay    40
short    10
Name: count, dtype: int64

Jom Corner (J Corner): 50 rows sampled
strata
english    20
malay      20
short      10
Name: count, dtype: int64

Total gold standard rows: 150


## 4. Run Pipeline on Gold Standard Rows (Candidate Labels)

This cell runs `run_pipeline()` on each of the 150 gold standard rows.
Output is candidate labels only — you manually verify every row after this.

**Note:** This will take a few minutes due to Google Translate API calls.

In [5]:
# Use on_no_match='other' so uncategorised aspects are kept, not dropped
categoriser = AspectCategoriser(nlp,on_no_match='other')

gold_records = []

for i, row in gold_df.iterrows():
    try:
        result = run_pipeline(row['review_text'], mode='auto', categoriser=categoriser)
        
        if result['results']:
            raw_aspect_lookup = {opinion: aspect for aspect, opinion in result['pairs']}
            for category, opinion_word in result['results']:
                gold_records.append({
                    'review_id':       row['review_id'],
                    'place_name':      row['place_name'],
                    'review_text':     row['review_text'],
                    'strata':          row['strata'],
                    'rating':          row['rating'],
                    'original_language': row['original_language'],
                    'aspect_category': category,
                    'opinion_word':    opinion_word,
                    'translated_text': result.get('translated') or result.get('normalised') or row['review_text'],
                    'was_translated':  result['was_translated'],
                    'raw_aspect': raw_aspect_lookup.get(opinion_word, None),
                    # --- Fields you fill in manually ---
                    'sentiment':       None,   # Positive / Negative
                    'is_manglish':     None,   # True / False
                    'human_verified':  False,
                    'annotation_notes': None,
                })
        else:
            # No pairs extracted — still keep row for annotation
            gold_records.append({
                'review_id':       row['review_id'],
                'place_name':      row['place_name'],
                'review_text':     row['review_text'],
                'strata':          row['strata'],
                'rating':          row['rating'],
                'original_language': row['original_language'],
                'aspect_category': None,
                'opinion_word':    None,
                'translated_text': result.get('translated') or result.get('normalised') or row['review_text'],
                'was_translated':  result['was_translated'],
                'sentiment':       None,
                'is_manglish':     None,
                'human_verified':  False,
                'annotation_notes': None,
            })
    except Exception as e:
        print(f'Error on review_id {row["review_id"]}: {e}')

    if (i + 1) % 10 == 0:
        print(f'Processed {i + 1} / {len(gold_df)} rows...')

gold_out = pd.DataFrame(gold_records)
print('\nGold standard candidate rows:', len(gold_out))
gold_out.head(5)

Processed 10 / 150 rows...
Processed 20 / 150 rows...
Processed 30 / 150 rows...
Processed 40 / 150 rows...
Processed 50 / 150 rows...
Processed 60 / 150 rows...
Processed 70 / 150 rows...
Processed 80 / 150 rows...
Processed 90 / 150 rows...
Processed 100 / 150 rows...
Processed 110 / 150 rows...
Processed 120 / 150 rows...
Processed 130 / 150 rows...
Processed 140 / 150 rows...
Processed 150 / 150 rows...

Gold standard candidate rows: 321


,review_id,place_name,review_text,strata,rating,original_language,aspect_category,opinion_word,translated_text,was_translated,raw_aspect,sentiment,is_manglish,human_verified,annotation_notes
0,ChZDSUhNMG9nS0VJQ0FnSURDakwtTGVREAE,Raihana One Bistro,So many choices and the nasi kandar is good,english,4,en,food,many,so many choices and the nasi kandar is good,False,choices,None,None,False,None
1,ChZDSUhNMG9nS0VJQ0FnSURDakwtTGVREAE,Raihana One Bistro,So many choices and the nasi kandar is good,english,4,en,food,good,so many choices and the nasi kandar is good,False,nasi kandar,None,None,False,None
2,ChdDSUhNMG9nS0VJQ0FnSUNTNE1yTTV3RRAB,Raihana One Bistro,I've tried mee goreng mamak and garlic naan. T...,english,4,en,food,other,i've tried mee goreng mamak and garlic naan. t...,False,dishes,None,None,False,None
3,ChZDSUhNMG9nS0VJQ0FnSURka3VIZk13EAE,Raihana One Bistro,Food is decent and affordable. Service could b...,english,4,en,food,normal,food is decent and affordable. service could b...,False,mamak restaurant,None,None,False,None
4,ChZDSUhNMG9nS0VJQ0FnSURka3VIZk13EAE,Raihana One Bistro,Food is decent and affordable. Service could b...,english,4,en,ambiance,normal,food is decent and affordable. service could b...,False,mamak restaurant,None,None,False,None


## 5. Save Pipeline A Output — Unverified

In [6]:
gold_out.to_csv('../data/malaysian_gold_standard_unverified.csv', index=False)
print('Saved: data/malaysian_gold_standard_unverified.csv')
print('Rows:', len(gold_out))

Saved: data/malaysian_gold_standard_unverified.csv
Rows: 321


## --- STOP HERE ---

Before running Pipeline B, manually annotate `malaysian_gold_standard_unverified.csv`:

1. Open the CSV in Excel.
2. For each row, fill in:
   - `sentiment` — `Positive` or `Negative` (your judgment, not the pipeline's)
   - `is_manglish` — `True` if the review mixes English and Malay, otherwise `False`
   - `annotation_notes` — optional, note anything unusual
   - `human_verified` — set to `True` once you've checked the row
3. Correct any wrong `aspect_category` or `opinion_word` values.
4. Save as `data/malaysian_gold_standard.csv`.

Then continue to Pipeline B below.

## 6. Pipeline B — Random Sample (300 per restaurant) + Full Pipeline Run

In [6]:
SAMPLE_SIZE = 300
dashboard_frames = []

for restaurant in restaurants:
    r_df = df[df['place_name'] == restaurant]
    n = min(SAMPLE_SIZE, len(r_df))
    sampled = r_df.sample(n, random_state=SEED)
    dashboard_frames.append(sampled)
    print(f'{restaurant}: {n} rows sampled')

dashboard_df = pd.concat(dashboard_frames).reset_index(drop=True)
print('\nTotal dashboard rows before pipeline:', len(dashboard_df))

Raihana One Bistro: 300 rows sampled
SABA Restaurant Cyberjaya: 300 rows sampled
Jom Corner (J Corner): 300 rows sampled

Total dashboard rows before pipeline: 900


## 7. Run Pipeline on Dashboard Rows

**Note:** This will take ~10-15 minutes for ~900 rows due to translation API calls.

In [7]:
import warnings
warnings.filterwarnings('ignore')

In [8]:
from scipy.sparse import hstack, csr_matrix
import pandas as pd
import numpy as np

CONFIDENCE_THRESHOLD = 0.65
categoriser = AspectCategoriser(nlp,on_no_match='other')

# Load model
import joblib
from sentence_transformers import SentenceTransformer

model = joblib.load('../models/model_proposed.pkl')
ohe   = joblib.load('../models/ohe_proposed.pkl')

transformer_name = open('../models/transformer_reference.txt').read().strip()
embedder = SentenceTransformer(transformer_name)

print('Models loaded OK')

transformer_name = open('../models/transformer_reference.txt').read().strip()
embedder = SentenceTransformer(transformer_name)

# Sentiment feature extractor — must match Notebook 03 exactly
def extract_sentiment_features_inference(text):
    pos_words      = {'good', 'great', 'excellent', 'amazing', 'love', 'best',
                      'delicious', 'fresh', 'wonderful', 'fantastic', 'perfect'}
    neg_words      = {'bad', 'terrible', 'awful', 'horrible', 'disappoint',
                      'slow', 'poor', 'bland', 'worst', 'rude', 'dirty'}
    negation_words = {'not', 'no', 'never', "n't", 'none', 'neither', 'nor', 'without'}
    contrast_words = {'but', 'however', 'though', 'although', 'despite', 'yet', 'while'}
    words             = str(text).split()
    has_pos           = int(any(w in pos_words for w in words))
    has_neg           = int(any(w in neg_words for w in words))
    mixed_sentiment   = int(has_pos and has_neg)
    negation_density  = sum(1 for w in words if w in negation_words) / max(len(words), 1)
    review_length     = len(words)
    has_contrast      = int(any(w in contrast_words for w in words))
    return csr_matrix(np.array([[mixed_sentiment, negation_density, review_length,
                                  has_contrast, has_pos, has_neg]], dtype=float))

dashboard_records = []

for i, row in dashboard_df.iterrows():
    try:
        result = run_pipeline(row['review_text'], mode='auto', categoriser=categoriser)

        if not result['results']:
            continue

        # Match Notebook 03 classify_review exactly
        sentence = result['translated'] if result['was_translated'] else row['review_text']

        for category, opinion_word in result['results']:
            aspect_input  = f'[ASPECT: {category.lower()}] {sentence}'
            embedding     = embedder.encode([aspect_input], convert_to_numpy=True)
            sentiment_vec = extract_sentiment_features_inference(sentence)
            aspect_vec    = ohe.transform([[category]])
            combined      = hstack([csr_matrix(embedding), sentiment_vec, aspect_vec])

            proba           = model.predict_proba(combined)[0]
            confidence      = float(proba.max())
            sentiment_label = model.classes_[proba.argmax()]

            dashboard_records.append({
                'review_id':           row['review_id'],
                'place_name':          row['place_name'],
                'review_text':         row['review_text'],
                'published_at':        row['published_at'],
                'aspect_category':     category,
                'opinion_word':        opinion_word,
                'sentiment':           sentiment_label,
                'confidence':          round(confidence, 4),
                'low_confidence_flag': confidence < CONFIDENCE_THRESHOLD,
            })
    except Exception as e:
        print(f'Error on review_id {row["review_id"]}: {e}')

    if (i + 1) % 50 == 0:
        print(f'Processed {i + 1} / {len(dashboard_df)} rows...')

dashboard_out = pd.DataFrame(dashboard_records)
print('\nDashboard rows after pipeline:', len(dashboard_out))
dashboard_out.head(5)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Models loaded OK


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Processed 50 / 900 rows...
Processed 150 / 900 rows...
Processed 250 / 900 rows...
Processed 300 / 900 rows...
Processed 350 / 900 rows...
Processed 400 / 900 rows...
Processed 450 / 900 rows...
Processed 500 / 900 rows...
Processed 550 / 900 rows...
Processed 600 / 900 rows...
Processed 650 / 900 rows...
Processed 700 / 900 rows...
Processed 750 / 900 rows...
Processed 800 / 900 rows...
Processed 850 / 900 rows...
Processed 900 / 900 rows...

Dashboard rows after pipeline: 2099


,review_id,place_name,review_text,published_at,aspect_category,opinion_word,sentiment,confidence,low_confidence_flag
0,ChdDSUhNMG9nS0VJQ0FnSUNqbnIzMGx3RRAB,Raihana One Bistro,order maggi goreng ayam tapi dia bagi maggi sa...,2 tahun yang lalu,price,foreign,negative,0.6343,True
1,ChdDSUhNMG9nS0VJQ0FnSUM2bHZid293RRAB,Raihana One Bistro,Nice and clean place..,4 tahun yang lalu,ambiance,nice,positive,0.9341,False
2,ChdDSUhNMG9nS0VJQ0FnSUM2bHZid293RRAB,Raihana One Bistro,Nice and clean place..,4 tahun yang lalu,ambiance,clean,positive,0.9341,False
3,ChdDSUhNMG9nS0VJQ0FnSURhejgtQXRBRRAB,Raihana One Bistro,Rasa dan harganya mmg berbaloi,4 tahun yang lalu,food,worth,positive,0.8659,False
4,ChdDSUhNMG9nS0VJQ0FnSURDODdEenRRRRAB,Raihana One Bistro,Good and clean place. Price OK. Went here freq...,5 tahun yang lalu,food,delicious,positive,0.8194,False


## 8. Save Pipeline B Output

In [9]:
dashboard_out.to_csv('../data/dashboard_data.csv', index=False)
print('Saved: data/dashboard_data.csv')
print('Total rows:', len(dashboard_out))
print()
print('Rows per restaurant:')
print(dashboard_out['place_name'].value_counts())
print()
print('Low confidence flags:', dashboard_out['low_confidence_flag'].sum())
print('Low confidence %:', round(dashboard_out['low_confidence_flag'].mean() * 100, 1), '%')
print()
print('Sentiment distribution:')
print(dashboard_out['sentiment'].value_counts())
print()
print('Aspect category distribution:')
print(dashboard_out['aspect_category'].value_counts())

Saved: data/dashboard_data.csv
Total rows: 2099

Rows per restaurant:
place_name
SABA Restaurant Cyberjaya    804
Jom Corner (J Corner)        658
Raihana One Bistro           637
Name: count, dtype: int64

Low confidence flags: 477
Low confidence %: 22.7 %

Sentiment distribution:
sentiment
positive    1396
negative     703
Name: count, dtype: int64

Aspect category distribution:
aspect_category
food        794
other       354
price       329
overall     211
ambiance    206
service     205
Name: count, dtype: int64
